# Task B batch #1 - same xattn architecture, 3-way head

**Goal.** Establish the first Task B baseline. This is deliberately the exact
`A160r` architecture and recipe that produced the best Task A single model, with
only the output dimension changed from 2 to 3.

- Backbone: `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`
- Shared comment/video encoder, 3 decoder blocks, no pooled-v
- `dropout 0.3`, `weight-decay 0.01`, `head-lr 1e-3`, `xattn-lr 1e-4`,
  `backbone-lr 2e-5`, `unfreeze-layers 3`, `layers 3`, `160/256`, 15 epochs,
  patience 5
- Task B labels: `no -> 0`, `yes_implicit -> 1`, `yes_explicit -> 2`
- Seeds 42/43/44, split-seed 42, video-cluster split
- Metric: 3-class macro-F1. Compare to majority `.2115` and TF-IDF `.4912`.

No class weights, no context dropout, no multi-task, no warm start in this batch.
Those are only worth testing after this number exists.

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/AmnO-O/Try.git'
PIN_COMMIT = 'cd35eadbbcee429ada54955b7d0786ad562ec09d'


def sh(cmd):
    print(f'$ {cmd}', flush=True)
    subprocess.run(cmd, shell=True, check=True)


# Re-run safe: after the first run the cwd is already <working>/Try.
if os.path.basename(os.path.abspath('.')) != 'Try':
    if not os.path.isdir('Try'):
        sh(f'git clone {REPO_URL}')
    os.chdir('Try')
if PIN_COMMIT:
    sh('git fetch origin main')
    sh(f'git checkout {PIN_COMMIT}')
sh('git log -1 --format="%H %cd %s" --date=iso')
sh('pip install -q -r requirements.txt')
print('cwd:', os.getcwd(), flush=True)

## 1. Data + GPU check

Attach `amnpham/lgbtqia` (Add Input), accelerator GPU T4 x2, Internet On. This
builds the split-42 train/val frames and saves the held-out labels, languages,
videos and IDs for the local analysis. The expected Task B val counts are
343 `no`, 175 `implicit`, 220 `explicit` (738 rows, 56 videos).

In [ ]:
import glob
import sys
import time

hits = sorted(glob.glob('/kaggle/input/**/StereoQueerEval_*_training.tsv', recursive=True))
assert hits, 'StereoQueerEval TSVs not found - attach the dataset (Add Input > amnpham/lgbtqia)'
os.environ['LGBT_DATA_DIR'] = os.path.dirname(hits[0])
os.environ['LGBT_WEIGHTS_DIR'] = os.path.abspath('weights')
os.makedirs('weights', exist_ok=True)
print('data dir:', os.environ['LGBT_DATA_DIR'], flush=True)

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), flush=True)
assert torch.cuda.is_available(), 'No GPU - set Settings > Accelerator > GPU T4 x2, then Run All'
print('gpu:', torch.cuda.get_device_name(0), flush=True)

sys.path.insert(0, os.path.abspath('src'))
import numpy as np
import pandas as pd
import data as data_mod
import model_xattn
from model import load_tokenizer
import train_xattn as tx

dev = tx.resolve_device('cuda')
df_all = data_mod.load_frame(
    languages=['EN', 'IT', 'NL'],
    label_column='hate_speech',
    label_mode='hate3',
)
df_train, df_val = data_mod.split_by_video(df_all, 0.1, 42)

labs = df_val['st_y'].astype(int).to_numpy()
counts = np.bincount(labs, minlength=3)
assert len(labs) == 738, f'unexpected val size: {len(labs)}'
assert counts.tolist() == [343, 175, 220], f'unexpected Task B val counts: {counts.tolist()}'
np.save('weights/val_labels_taskb.npy', labs)
np.save('weights/val_langs_taskb.npy', df_val['lang'].to_numpy(dtype=str))
np.save('weights/val_videos_taskb.npy', df_val['yt_title'].to_numpy(dtype=str))
np.save('weights/val_ids_taskb.npy', df_val['StereoQueerEval_id'].to_numpy(dtype=str))
quads = (df_val['stereotype'].astype(str) + '|' + df_val['hate_speech'].astype(str)).to_numpy(dtype=str)
np.save('weights/val_quads_taskb.npy', quads)

print('val:', len(df_val), 'rows |', df_val['yt_title'].nunique(), 'videos', flush=True)
print('  Task B counts:', dict(zip(('no', 'implicit', 'explicit'), counts.tolist())), flush=True)
print('  train counts:',
      dict(zip(('no', 'implicit', 'explicit'),
               np.bincount(df_train['st_y'].astype(int), minlength=3).tolist())), flush=True)
print('  val quadrant counts:', dict(pd.Series(quads).value_counts()), flush=True)

## 2. Run list and helpers

Exact A160 recipe. The only change relative to Task A is `--task b`, which maps
the three-way `hate_speech` label and sets `num_classes=3`. The training script
also writes the best checkpoint's held-out probabilities for ensembling.

In [ ]:
BASE = dict(
    dropout=0.3, weight_decay=0.01, head_lr=1e-3, xattn_lr=1e-4,
    backbone_lr=2e-5, unfreeze_layers=3, layers=3, shared_encoder=True,
    pooled_v=False, freeze_epochs=0, pool_init_scale=0.02,
    max_len=160, video_max_len=256, no_amp=False,
    backbone=model_xattn.DEFAULT_BACKBONE,
)

RUNS = [dict(BASE, tag=f'TB_s{s}', seed=s, split_seed=42) for s in (42, 43, 44)]


def train_one(r):
    hist = f"weights/{r['tag']}_history.csv"
    out = f"weights/{r['tag']}.pt"
    probs = f"weights/{r['tag']}_val_probs.npy"
    if os.path.exists(hist) and os.path.exists(out):
        print(f"-- skip (checkpoint + history exist): {r['tag']}", flush=True)
        return True
    cmd = (
        'python src/train_xattn.py --task b --languages EN IT NL '
        '--epochs 15 --patience 5 --batch-size 32 '
        f"--backbone {r['backbone']} "
        f"--max-len {r['max_len']} --video-max-len {r['video_max_len']} "
        f"--dropout {r['dropout']} --weight-decay {r['weight_decay']} "
        f"--head-lr {r['head_lr']} --xattn-lr {r['xattn_lr']} --backbone-lr {r['backbone_lr']} "
        f"--unfreeze-layers {r['unfreeze_layers']} --layers {r['layers']} "
        f"--pool-init-scale {r['pool_init_scale']} --freeze-epochs {r['freeze_epochs']} "
        f"--context-dropout 0 --no-pooled-v --shared-encoder "
        f"--split-seed {r['split_seed']} --seed {r['seed']} "
        f"--out {out} --history {hist} --probs-out {probs}"
    )
    print(f'$ {cmd}', flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'}).returncode
    print(f"== {r['tag']}: exit {rc} after {(time.time() - t0) / 60:.1f} min", flush=True)
    if rc != 0 or not os.path.exists(hist):
        print(f'!! {r["tag"]} failed', flush=True)
        return False
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    print(f"   {r['tag']}: best epoch {int(h.loc[bi, 'epoch'])} | "
          f"val_macro_f1 {float(h.loc[bi, 'val_macro_f1']):.4f} | "
          f"acc {float(h.loc[bi, 'val_acc']):.4f} | {len(h)} epochs", flush=True)
    return True


def ensure_probs(r):
    """Recover held-out probabilities if a resumed run predates --probs-out."""
    p = f"weights/{r['tag']}_val_probs.npy"
    ckpt = f"weights/{r['tag']}.pt"
    if os.path.exists(p):
        return
    if not os.path.exists(ckpt):
        print(f'-- no checkpoint for {r["tag"]}', flush=True)
        return
    net = model_xattn.build_model(
        r['backbone'], unfreeze_layers=r['unfreeze_layers'],
        heads=4, layers=r['layers'], dropout=r['dropout'], ffn_mult=4,
        use_pooled_v=r['pooled_v'], separate_encoders=not r['shared_encoder'],
        keep_frozen_dropout=False, pool_init_scale=r['pool_init_scale'],
        num_classes=3,
    ).to(dev)
    net.load_state_dict(torch.load(ckpt, map_location=dev))
    net.eval()
    tok_r = load_tokenizer(r['backbone'])
    if tok_r.pad_token is None:
        tok_r.pad_token = tok_r.eos_token or tok_r.sep_token
    loader = tx.make_loader(df_val, tok_r, 32, r['max_len'], r['video_max_len'])
    o = tx.collect(net, loader, dev, use_amp=True)
    np.save(p, o['probs'])
    m = tx.metrics_from_taskb(o['preds'], o['labels'])
    print(f"   {r['tag']}: recovered probs | macro_f1 {m['macro_f1']:.4f}", flush=True)
    del net
    torch.cuda.empty_cache()


def zip_small():
    import zipfile
    small = [f for f in sorted(os.listdir('weights'))
             if f.endswith(('.csv', '.npy', '.json'))]
    with zipfile.ZipFile('weights/results_small.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in small:
            zf.write(os.path.join('weights', f), f)
    print(f'wrote weights/results_small.zip ({len(small)} files)', flush=True)


print('planned:', len(RUNS), 'runs |', [r['tag'] for r in RUNS], flush=True)

## 3. Train the three seeds

About 20-30 min on T4 x2; resume-safe. Partial small results are zipped after
every run, so a disconnected session keeps the histories and probabilities.

In [ ]:
t0 = time.time()
for i, r in enumerate(RUNS, 1):
    print(f'=== [{i}/{len(RUNS)}] {r["tag"]} (elapsed {(time.time() - t0) / 60:.0f} min) ===', flush=True)
    try:
        train_one(r)
    except Exception as e:                           # noqa: BLE001
        print(f'!! train_one({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        ensure_probs(r)
    except Exception as e:                           # noqa: BLE001
        print(f'!! ensure_probs({r["tag"]}) raised: {type(e).__name__}: {e}', flush=True)
    try:
        zip_small()
    except Exception as e:                           # noqa: BLE001
        print(f'!! zip_small raised: {type(e).__name__}: {e}', flush=True)
print(f'all {len(RUNS)} runs finished in {(time.time() - t0) / 3600:.2f} h', flush=True)

## 4. Baseline summary

Per-seed history, reloaded macro-F1, per-class F1, confusion matrix, prediction
distribution, and the 3-seed equal-weight ensemble. The raw arrays are saved for
the local paired video-cluster bootstrap; do not tune thresholds or weights on
these 738 rows.

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

labs = np.load('weights/val_labels_taskb.npy')
langs = np.load('weights/val_langs_taskb.npy', allow_pickle=True)
names = ('no', 'implicit', 'explicit')
rows = []
probs_by_tag = {}

for r in RUNS:
    hist = f"weights/{r['tag']}_history.csv"
    p = f"weights/{r['tag']}_val_probs.npy"
    if not (os.path.exists(hist) and os.path.exists(p)):
        print(f'missing history/probs: {r["tag"]}', flush=True)
        continue
    h = pd.read_csv(hist)
    bi = h['val_macro_f1'].idxmax()
    probs = np.load(p)
    pred = probs.argmax(1)
    m = tx.metrics_from_taskb(pred.tolist(), labs.tolist())
    cm = confusion_matrix(labs, pred, labels=[0, 1, 2])
    probs_by_tag[r['tag']] = probs
    rows.append(dict(
        tag=r['tag'], seed=r['seed'],
        best_epoch=int(h.loc[bi, 'epoch']),
        best_hist_f1=float(h.loc[bi, 'val_macro_f1']),
        reload_f1=float(m['macro_f1']),
        acc=float(m['acc']),
        f1_no=float(m['f1_no']),
        f1_implicit=float(m['f1_implicit']),
        f1_explicit=float(m['f1_explicit']),
        pred_no=float((pred == 0).mean()),
        pred_implicit=float((pred == 1).mean()),
        pred_explicit=float((pred == 2).mean()),
    ))
    print(f"{r['tag']}: best ep {int(h.loc[bi, 'epoch']):>2} | "
          f"hist {float(h.loc[bi, 'val_macro_f1']):.4f} | reload {m['macro_f1']:.4f} | "
          f"acc {m['acc']:.4f} | F1 no {m['f1_no']:.3f} imp {m['f1_implicit']:.3f} "
          f"exp {m['f1_explicit']:.3f} | pred {pred.mean():.3f} implicit "
          f"{(pred == 1).mean():.3f}", flush=True)
    print(tx.format_confusion_matrix_taskb(labs, pred), flush=True)

summ = pd.DataFrame(rows).sort_values('seed')
summ.to_csv('weights/taskB_batch1_summary.csv', index=False)
print()
print(summ.to_string(index=False), flush=True)

if len(probs_by_tag) == 3:
    ens = np.mean([probs_by_tag[r['tag']] for r in RUNS], axis=0)
    ep = ens.argmax(1)
    em = tx.metrics_from_taskb(ep.tolist(), labs.tolist())
    print()
    print(f"3-seed ensemble: macro_f1 {em['macro_f1']:.4f} | acc {em['acc']:.4f} | "
          f"F1 no {em['f1_no']:.4f} imp {em['f1_implicit']:.4f} exp {em['f1_explicit']:.4f}",
          flush=True)
    print(tx.format_confusion_matrix_taskb(labs, ep), flush=True)
    print('  ensemble predicted distribution:',
          dict(zip(names, (np.bincount(ep, minlength=3) / len(ep)).round(3).tolist())), flush=True)
    print('--- PER LANGUAGE (ensemble) ---', flush=True)
    for lang in sorted(set(langs.tolist())):
        msk = langs == lang
        ml = tx.metrics_from_taskb(ep[msk].tolist(), labs[msk].tolist())
        print(f"  [{lang}] n {int(msk.sum())} | macro_f1 {ml['macro_f1']:.4f} | "
              f"F1 no {ml['f1_no']:.3f} imp {ml['f1_implicit']:.3f} exp {ml['f1_explicit']:.3f}",
              flush=True)
else:
    print(f'only {len(probs_by_tag)}/3 seeds have probabilities; no ensemble', flush=True)

maj = np.zeros_like(labs)
maj_f1 = f1_score(labs, maj, labels=[0, 1, 2], average='macro', zero_division=0)
mean_best = float(summ['best_hist_f1'].mean()) if len(summ) else float('nan')
print()
print(f'baselines: majority {maj_f1:.4f} | recorded TF-IDF .4912', flush=True)
print(f'xattn Task B: mean best {mean_best:.4f}', flush=True)
if len(probs_by_tag) == 3:
    print(f'xattn Task B: 3-seed ensemble {em["macro_f1"]:.4f}', flush=True)
    print('gate: >= .55 -> proceed to class weights / A-warm-start / LLM; '
          '<= .491 -> diagnose before more compute', flush=True)
zip_small()
print('TASKB BATCH1 COMPLETE', flush=True)